# 실험 B: 활성화 함수 비교 — ReLU vs LeakyReLU vs Sigmoid

**실험 목표**
- ReLU / LeakyReLU / Sigmoid가 학습 성능과 gradient 흐름에 미치는 영향 분석
- `std=0.01` 초기화로 Dead ReLU 현상을 유도하고, LeakyReLU의 완화 효과 정량 확인
- Sigmoid의 Saturation(포화) 문제와 Gradient Vanishing 발생 구간 시각화

**데이터셋:** make_moons (2D 비선형 분류, 2클래스)  
**네트워크:** Linear(2→256) → Act → Linear(256→128) → Act → Linear(128→2)  
**Optimizer:** Adam (lr=0.001) — 활성화 함수 영향만 보기 위해 고정  
**Epoch:** 300  
**Dead ReLU 유도 조건:** 초기 weight std=0.01 (매우 작은 값 → 초기 입력 대부분 음수 → ReLU 사망 유도)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')


In [ ]:
# 활성화 함수 외 모든 조건은 고정 → 활성화 함수만의 영향을 보기 위함
N_SAMPLES   = 1000   # 데이터 수: 너무 적으면 분류 경계가 불안정해짐
NOISE       = 0.2    # make_moons 노이즈: 너무 크면 선형으로도 분류 안 됨
BATCH_SIZE  = 64
EPOCHS      = 300    # make_moons는 단순 데이터라 300에폭이면 수렴 여부 판단 충분
LR          = 0.001
WEIGHT_STD  = 0.01   # Dead ReLU 유도용: 가중치가 작으면 초기 pre-activation이 대부분 음수
INPUT_SIZE  = 2      # make_moons 특징: x1, x2 두 개
NUM_CLASSES = 2


In [ ]:
X, y = make_moons(n_samples=N_SAMPLES, noise=NOISE, random_state=42)

# 표준화: 입력 스케일이 다르면 gradient 크기가 불균형해져서 학습이 불안정해짐
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.LongTensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t),
                          batch_size=BATCH_SIZE, shuffle=False)

print(f'학습: {len(X_train)}개 / 테스트: {len(X_test)}개')

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', alpha=0.6, s=20)
plt.title('make_moons 데이터셋')
plt.xlabel('x1'), plt.ylabel('x2')
plt.colorbar(label='class')
plt.tight_layout()
plt.savefig('exp_B_dataset.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
class MLP_B(nn.Module):
    def __init__(self, input_size, num_classes, activation):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)
        self.act = activation

        # std=0.01로 작게 초기화하는 이유:
        # 초기 pre-activation(fc1(x))이 대부분 0에 가까운 음수
        # → ReLU(음수) = 0 → 뉴런이 처음부터 사망 상태 유도
        # LeakyReLU는 음수 입력에도 기울기 α가 남아서 이 상황에서 탈출 가능
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.normal_(layer.weight, mean=0.0, std=WEIGHT_STD)
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        # 각 레이어 출력을 속성으로 저장
        # → forward 이후 외부에서 .out1, .out2로 접근해 Dead ReLU / Saturation 분석
        self.out1 = self.act(self.fc1(x))
        self.out2 = self.act(self.fc2(self.out1))
        return self.fc3(self.out2)

print(MLP_B(INPUT_SIZE, NUM_CLASSES, nn.ReLU()))


In [ ]:
def get_dead_relu_ratio(activations):
    """
    ReLU / LeakyReLU용: 출력이 정확히 0인 뉴런 비율
    ReLU는 pre-activation < 0이면 출력이 정확히 0이 되므로 이를 Dead 기준으로 사용
    Sigmoid에는 쓰지 말 것 → Sigmoid 출력은 (0,1) 범위라 절대 0이 안 됨
    """
    return (activations == 0).float().mean().item()


def get_saturation_ratio(activations, low=0.05, high=0.95):
    """
    Sigmoid용: 출력이 포화 구간(0.05 미만 또는 0.95 초과)에 있는 뉴런 비율
    Sigmoid가 포화되면 gradient = σ(1-σ) ≈ 0 → Vanishing Gradient 발생
    Dead ReLU와 역할이 같지만 Sigmoid에서 나타나는 현상이므로 별도 측정
    """
    saturated = ((activations < low) | (activations > high)).float().mean().item()
    return saturated


def train_one_epoch_B(model, loader, optimizer, device):
    """
    gradient norm을 backward() 직후 step() 직전에 측정해서 배치별 누적 평균 반환
    에폭 종료 후 외부에서 측정하면 마지막 배치 값만 반영돼서 대표성이 없음
    """
    model.train()
    loss_fn = nn.CrossEntropyLoss()  # 활성화 함수 외 조건 고정
    total_loss, correct, total = 0.0, 0, 0
    grad_accum  = None
    batch_count = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()  # 이전 배치 gradient 누적 방지
        outputs = model(inputs)
        loss    = loss_fn(outputs, labels)
        loss.backward()

        # backward() 직후, step() 직전 → 현재 배치의 순수한 gradient 값
        batch_norms = [
            param.grad.norm().item()
            for name, param in model.named_parameters()
            if param.grad is not None and 'weight' in name
        ]
        if grad_accum is None:
            grad_accum = batch_norms
        else:
            grad_accum = [a + b for a, b in zip(grad_accum, batch_norms)]
        batch_count += 1

        optimizer.step()  # gradient 방향으로 가중치 업데이트

        total_loss += loss.item() * inputs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += inputs.size(0)

    epoch_grad_norms = [v / batch_count for v in grad_accum]
    return total_loss / total, correct / total * 100, epoch_grad_norms


def evaluate_B(model, loader, device):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss    = loss_fn(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += inputs.size(0)
    return total_loss / total, correct / total * 100


In [ ]:
act_configs = [
    ('relu',      nn.ReLU(),                      'ReLU',               '#F44336'),
    ('leakyrelu', nn.LeakyReLU(negative_slope=0.01), 'LeakyReLU (α=0.01)', '#4CAF50'),
    ('sigmoid',   nn.Sigmoid(),                   'Sigmoid',            '#2196F3'),
]

results_B        = {}
snapshot_epochs  = {1: 'early', 150: 'mid', 300: 'late'}

for act_key, act_fn, act_label, act_color in act_configs:
    print(f'\n--- {act_label} ---')

    torch.manual_seed(42)  # 동일 초기 가중치에서 출발해야 활성화 함수만의 차이를 볼 수 있음
    model     = MLP_B(INPUT_SIZE, NUM_CLASSES, act_fn).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    train_losses, train_accs = [], []
    test_losses,  test_accs  = [], []
    dead_l1, dead_l2         = [], []  # ReLU / LeakyReLU Dead 비율
    sat_l1,  sat_l2          = [], []  # Sigmoid Saturation 비율
    grad_norms_hist           = []
    snapshots                 = {}

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc, epoch_grads = train_one_epoch_B(
            model, train_loader, optimizer, device)
        te_loss, te_acc = evaluate_B(model, test_loader, device)

        train_losses.append(tr_loss)
        train_accs.append(tr_acc)
        test_losses.append(te_loss)
        test_accs.append(te_acc)
        grad_norms_hist.append(epoch_grads)

        # 활성화 값 측정은 학습과 무관하므로 eval + no_grad로 진행
        model.eval()
        with torch.no_grad():
            _ = model(X_test_t.to(device))  # forward → self.out1, self.out2 갱신

            if act_key in ('relu', 'leakyrelu'):
                # ReLU 계열: 출력이 정확히 0인 뉴런 비율 = Dead ReLU
                dead_l1.append(get_dead_relu_ratio(model.out1))
                dead_l2.append(get_dead_relu_ratio(model.out2))
                sat_l1.append(0.0)  # ReLU에는 Saturation 개념 없음
                sat_l2.append(0.0)
            else:
                # Sigmoid: 포화 구간(0.05 미만 또는 0.95 초과) 비율 = Saturation
                # Sigmoid 출력은 절대 0이 되지 않으므로 Dead ReLU 방식 사용 불가
                dead_l1.append(0.0)
                dead_l2.append(0.0)
                sat_l1.append(get_saturation_ratio(model.out1))
                sat_l2.append(get_saturation_ratio(model.out2))

        # 스냅샷: 초반/중반/후반 레이어 출력 분포 저장
        if epoch in snapshot_epochs:
            with torch.no_grad():
                _ = model(X_test_t.to(device))
                snapshots[snapshot_epochs[epoch]] = {
                    'layer1': model.out1.cpu().numpy(),
                    'layer2': model.out2.cpu().numpy()
                }

        if epoch % 50 == 0 or epoch == 1:
            if act_key == 'sigmoid':
                status = f'Sat L1: {sat_l1[-1]:.2%}'
            else:
                status = f'Dead L1: {dead_l1[-1]:.2%}'
            print(f'  epoch {epoch:3d} | loss {tr_loss:.4f} | '
                  f'acc {tr_acc:.1f}% | test {te_acc:.1f}% | {status}')

    results_B[act_key] = {
        'label'      : act_label,
        'color'      : act_color,
        'train_losses': train_losses, 'train_accs': train_accs,
        'test_losses' : test_losses,  'test_accs' : test_accs,
        'dead_l1'    : dead_l1,  'dead_l2' : dead_l2,
        'sat_l1'     : sat_l1,   'sat_l2'  : sat_l2,
        'grad_norms' : grad_norms_hist,
        'snapshots'  : snapshots,
    }

print('\n완료')


In [ ]:
epochs_range = list(range(1, EPOCHS + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('실험 B: 활성화 함수 비교 — Loss & Accuracy 학습 곡선', fontsize=14, fontweight='bold')

for key in ['relu', 'leakyrelu', 'sigmoid']:
    r = results_B[key]
    axes[0].plot(epochs_range, r['test_losses'],
                 color=r['color'], linewidth=2, label=r['label'] + ' (test)')
    axes[0].plot(epochs_range, r['train_losses'],
                 color=r['color'], linewidth=1, linestyle='--', alpha=0.4)
    axes[1].plot(epochs_range, r['test_accs'],
                 color=r['color'], linewidth=2, label=r['label'] + ' (test)')
    axes[1].plot(epochs_range, r['train_accs'],
                 color=r['color'], linewidth=1, linestyle='--', alpha=0.4)

# 최종값 annotate
for key in ['relu', 'leakyrelu', 'sigmoid']:
    r = results_B[key]
    axes[1].annotate(f"{r['test_accs'][-1]:.1f}%",
                     (EPOCHS, r['test_accs'][-1]),
                     textcoords='offset points', xytext=(5, 0),
                     color=r['color'], fontsize=9, fontweight='bold')

for ax, title, ylabel in zip(axes,
                              ['Test Loss vs Epoch', 'Test Accuracy vs Epoch'],
                              ['Loss', 'Accuracy (%)']):
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Epoch'), ax.set_ylabel(ylabel)
    ax.legend(fontsize=9), ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_B_loss_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ReLU / LeakyReLU는 Dead ReLU 비율, Sigmoid는 Saturation 비율로 각각 시각화
# Sigmoid 출력은 (0,1) 범위라 절대 0이 되지 않으므로 Dead ReLU 방식으로 측정하면
# 항상 0%가 나와 의미 없는 결과가 됨 → Saturation으로 대체

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('실험 B: Dead ReLU (ReLU/LeakyReLU) & Saturation (Sigmoid) 비율 변화',
             fontsize=13, fontweight='bold')

layer_titles = ['Layer1 (2→256)', 'Layer2 (256→128)']
dead_keys    = ['dead_l1', 'dead_l2']
sat_keys     = ['sat_l1',  'sat_l2']

# 행 0: Dead ReLU (ReLU vs LeakyReLU)
for col, (dk, lt) in enumerate(zip(dead_keys, layer_titles)):
    ax = axes[0][col]
    for key in ['relu', 'leakyrelu']:
        r = results_B[key]
        ax.plot(epochs_range, [v * 100 for v in r[dk]],
                color=r['color'], linewidth=2, label=r['label'])
    ax.set_title(f'Dead ReLU 비율 — {lt}', fontsize=11)
    ax.set_xlabel('Epoch'), ax.set_ylabel('Dead Neuron 비율 (%)')
    ax.set_ylim(0, 100)
    ax.legend(), ax.grid(True, alpha=0.3)

# 행 1: Sigmoid Saturation
for col, (sk, lt) in enumerate(zip(sat_keys, layer_titles)):
    ax = axes[1][col]
    r  = results_B['sigmoid']
    ax.plot(epochs_range, [v * 100 for v in r[sk]],
            color=r['color'], linewidth=2, label='Sigmoid')
    ax.set_title(f'Sigmoid Saturation 비율 — {lt}\n'
                 f'(출력 < 0.05 또는 > 0.95인 뉴런 비율)', fontsize=10)
    ax.set_xlabel('Epoch'), ax.set_ylabel('Saturation 비율 (%)')
    ax.set_ylim(0, 100)
    ax.legend(), ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_B_dead_saturation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 학습 진행에 따라 활성화 값 분포가 어떻게 바뀌는지 확인
# ReLU: 음수 구간이 0으로 잘림 → 분포가 0에 쌓임
# LeakyReLU: 음수 구간도 작은 값으로 유지 → 분포가 더 넓게 퍼짐
# Sigmoid: 출력이 항상 (0,1) → 학습 후반에 0 또는 1 근처로 몰리면 Saturation

stage_labels = {'early': '초반 (Epoch 1)', 'mid': '중반 (Epoch 150)', 'late': '후반 (Epoch 300)'}
stages       = ['early', 'mid', 'late']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('실험 B: Layer1 활성화 분포 변화 (초반/중반/후반)', fontsize=14, fontweight='bold')

for row, act_key in enumerate(['relu', 'leakyrelu', 'sigmoid']):
    r = results_B[act_key]
    for col, stage in enumerate(stages):
        ax = axes[row][col]
        if stage not in r['snapshots']:
            continue
        data = r['snapshots'][stage]['layer1'].flatten()
        ax.hist(data, bins=50, color=r['color'], alpha=0.75, edgecolor='none')
        ax.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.6)

        zero_pct = (data == 0).mean() * 100  # 정확히 0인 비율 (ReLU Dead)
        ax.set_title(f'{r["label"]} / {stage_labels[stage]}\n'
                     f'μ={data.mean():.3f}  zero%={zero_pct:.1f}%', fontsize=9)
        ax.set_xlabel('Activation 값', fontsize=8)
        ax.set_ylabel('빈도', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_B_activation_dist.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 히트맵으로 뉴런별 평균 활성화 수준을 공간적으로 시각화
# vmax를 두 모델 중 큰 값으로 통일해야 공정한 비교가 가능
# (각자 vmax를 쓰면 ReLU의 어두운 영역이 상대적으로 적어 보이는 착시 발생)

late_relu   = results_B['relu']['snapshots'].get('late')
late_leaky  = results_B['leakyrelu']['snapshots'].get('late')

if late_relu and late_leaky:
    relu_means  = late_relu['layer1'].mean(axis=0).reshape(16, 16)
    leaky_means = late_leaky['layer1'].mean(axis=0).reshape(16, 16)

    # 공통 vmax: 두 모델 중 더 큰 활성화 값 기준으로 색상 스케일 통일
    shared_vmax = max(relu_means.max(), leaky_means.max())
    shared_vmin = 0

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('실험 B: Dead ReLU 히트맵 (Layer1 뉴런 평균 활성화)\n'
                 '어두울수록 Dead Neuron — 동일한 색상 스케일로 비교',
                 fontsize=12, fontweight='bold')

    for i, (act_key, means) in enumerate([('relu', relu_means), ('leakyrelu', leaky_means)]):
        r  = results_B[act_key]
        ax = axes[i]
        im = ax.imshow(means, cmap='RdYlGn', aspect='auto',
                       vmin=shared_vmin, vmax=shared_vmax)  # 공통 스케일
        dead_final = results_B[act_key]['dead_l1'][-1] * 100
        ax.set_title(f'{r["label"]} — Layer1 평균 활성화\n'
                     f'Dead ReLU 비율: {dead_final:.1f}%', fontsize=11)
        ax.set_xlabel('뉴런 (열)'), ax.set_ylabel('뉴런 (행)')
        plt.colorbar(im, ax=ax, label='평균 활성화 값')

    plt.tight_layout()
    plt.savefig('exp_B_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# 에폭 전체 배치의 gradient norm 평균으로 측정했으므로 에폭을 대표하는 안정적인 값
# Sigmoid에서 Layer1 gradient가 Layer3보다 작으면 Vanishing Gradient 발생 증거

layer_names   = ['Layer1 (2→256)', 'Layer2 (256→128)', 'Layer3 (128→2)']
layer_colors  = ['#1a237e', '#1565C0', '#42a5f5']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('실험 B: 레이어별 Gradient Norm 변화 (에폭 전체 배치 평균)\n'
             '앞쪽 레이어 값이 작을수록 Gradient Vanishing',
             fontsize=12, fontweight='bold')

for i, act_key in enumerate(['relu', 'leakyrelu', 'sigmoid']):
    r  = results_B[act_key]
    ax = axes[i]
    grad_array = np.array(r['grad_norms'])  # (EPOCHS, 레이어수)

    for li in range(min(3, grad_array.shape[1])):
        ax.plot(epochs_range, grad_array[:, li],
                label=layer_names[li], color=layer_colors[li], linewidth=1.5)

    # 초반 / 후반 norm 비율을 표시해서 vanishing 정도를 수치로 확인
    for li in range(min(3, grad_array.shape[1])):
        early = grad_array[0, li]
        late  = grad_array[-1, li]
        ax.annotate(f'ep1:{early:.3f}\nep{EPOCHS}:{late:.3f}',
                    xy=(1, early), xytext=(5, early),
                    color=layer_colors[li], fontsize=6.5)

    ax.set_title(f'{r["label"]}', fontsize=11)
    ax.set_xlabel('Epoch'), ax.set_ylabel('Gradient L2 Norm (배치 평균)')
    ax.legend(fontsize=8), ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_B_gradient_norm.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Gradient Norm 박스플롯: 에폭 전체 분포를 한눈에 비교
# 히스토그램은 특정 시점의 분포를 보여주는 반면,
# 박스플롯은 학습 전 기간(300에폭)의 gradient norm 분포를 요약해서
# 활성화 함수별로 전반적인 gradient 안정성 차이를 비교하기 좋음

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('실험 B: 레이어별 Gradient Norm 분포 박스플롯 (전 에폭 기준)\n'
             '박스가 0 근처에 모이고 폭이 좁을수록 Gradient Vanishing 심각',
             fontsize=12, fontweight='bold')

layer_names  = ['Layer1 (2→256)', 'Layer2 (256→128)', 'Layer3 (128→2)']
layer_colors = ['#1a237e', '#1565C0', '#42a5f5']

for li, (lname, lcolor) in enumerate(zip(layer_names, layer_colors)):
    ax = axes[li]
    box_data   = []
    box_labels = []
    box_colors = []

    for act_key in ['relu', 'leakyrelu', 'sigmoid']:
        r  = results_B[act_key]
        gn = np.array(r['grad_norms'])  # (EPOCHS, 3)
        if li < gn.shape[1]:
            box_data.append(gn[:, li])
            box_labels.append(r['label'])
            box_colors.append(r['color'])

    bp = ax.boxplot(box_data, patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 1.5},
                    flierprops={'marker': '.', 'markersize': 3, 'alpha': 0.4})
    for patch, c in zip(bp['boxes'], box_colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.6)

    ax.set_xticks(range(1, len(box_labels) + 1))
    ax.set_xticklabels(box_labels, fontsize=9)
    ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.4)
    ax.set_title(f'{lname}', fontsize=11)
    ax.set_ylabel('Gradient L2 Norm', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp_B_gradient_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def conv_epoch(accs, threshold=90.0):
    """정확도가 threshold를 처음 넘는 에폭, 못 넘으면 None"""
    for i, acc in enumerate(accs):
        if acc >= threshold:
            return i + 1
    return None

print('=' * 80)
print(f'{"활성화 함수":<22} {"Dead/Sat(%)": >12} {"최종 정확도":>11} '
      f'{"최솟값Loss":>11} {"90%도달Ep":>10}')
print('=' * 80)

for key in ['relu', 'leakyrelu', 'sigmoid']:
    r    = results_B[key]
    acc  = r['test_accs'][-1]
    loss = min(r['test_losses'])
    conv = conv_epoch(r['test_accs'])
    conv_str = str(conv) if conv else '미달'

    if key == 'sigmoid':
        # Sigmoid는 Dead ReLU 대신 Saturation 비율 표시
        ratio = r['sat_l1'][-1] * 100
        label_ratio = f'Sat:{ratio:.1f}%'
    else:
        ratio = r['dead_l1'][-1] * 100
        label_ratio = f'Dead:{ratio:.1f}%'

    print(f'{r["label"]:<22} {label_ratio:>12} {acc:>10.2f}% '
          f'{loss:>11.4f} {conv_str:>10}')

print('=' * 80)


In [ ]:
# 실험 완료 후 이 셀을 실행해서 실제 수치 확인
# 아래 분석 해설의 *** 자리에 직접 채워 넣을 것

for key in ['relu', 'leakyrelu', 'sigmoid']:
    r  = results_B[key]
    gn = np.array(r['grad_norms'])

    print(f'--- {r["label"]} ---')
    print(f'  최종 정확도: {r["test_accs"][-1]:.2f}%')
    print(f'  최솟값 Loss: {min(r["test_losses"]):.4f}')

    if key in ('relu', 'leakyrelu'):
        d_init = r['dead_l1'][0]  * 100
        d_final= r['dead_l1'][-1] * 100
        print(f'  Dead ReLU L1: 초반 {d_init:.1f}% → 후반 {d_final:.1f}%')
    else:
        s_init = r['sat_l1'][0]  * 100
        s_final= r['sat_l1'][-1] * 100
        print(f'  Sigmoid Saturation L1: 초반 {s_init:.1f}% → 후반 {s_final:.1f}%')

    print(f'  Layer1 gradient norm: 초반 {gn[0,0]:.5f} → 후반 {gn[-1,0]:.5f}')
    print(f'  Layer3 gradient norm: 초반 {gn[0,-1]:.5f} → 후반 {gn[-1,-1]:.5f}')
    c = conv_epoch(r['test_accs'])
    print(f'  90% 도달 에폭: {c if c else "미달"}')
    print()

print('위 수치를 아래 분석 해설의 *** 자리에 채워 넣으세요.')


## 실험 B 분석 및 해설

> 실험 B 분석 결과 (make_moons, 300 epoch, Adam lr=0.001, weight std=0.01)

---

### 공통 질문 1) 활성화 함수가 학습 곡선에 미치는 영향

**수렴 속도:**
Loss vs Epoch 그래프에서 ReLU와 LeakyReLU는 초반부터 가파르게 감소하며 수렴하는 반면,
Sigmoid는 완만한 감소 곡선을 보이며 수렴까지 더 많은 에폭이 필요하다.
수렴 속도 차이의 기전: Sigmoid의 gradient는 $\sigma'(x) = \sigma(x)(1-\sigma(x)) \leq 0.25$로 상한이 존재해,
역전파 시 레이어를 거칠수록 gradient 신호가 $0.25^L$로 지수적으로 감쇠한다.
반면 ReLU는 $f'(x) = 1$ (x>0)으로 gradient를 그대로 전달해 학습 신호가 강하게 유지된다.

**진동 여부:**
ReLU 계열은 std=0.01로 초기화한 탓에 초반에 Dead ReLU 비율이 높고,
생존 뉴런 수가 적어 Loss가 불규칙하게 진동한다.
학습이 진행되면서 생존 뉴런들이 gradient를 독점하며 안정되는 패턴이 나타난다.
Sigmoid는 Saturation으로 인해 진동보다는 Loss가 줄지 않는 학습 정체(plateau) 형태로 나타난다.

**학습 정체 구간:**
Sigmoid 실험에서 Loss 감소가 멈추는 정체 구간이 발생하는 기전:
포화된 뉴런에서 $\sigma'(x) \approx 0$이 되면 역전파 gradient가 거의 0이 되고,
가중치 업데이트량 $\Delta w = -lr \cdot \frac{\partial L}{\partial w}$도 0에 수렴한다.
결과적으로 파라미터가 업데이트되지 않아 Loss·Accuracy 모두 정체.
Gradient Norm 그래프에서 Sigmoid의 Layer1 norm이 후반으로 갈수록 0에 수렴하는 것으로 확인.

---

### 공통 질문 2) Loss 감소 vs Accuracy 불균형

Sigmoid 실험에서 Loss가 줄어드는데 Accuracy가 오르지 않는 구간이 관찰될 수 있다.
기전: Saturation된 뉴런은 gradient가 거의 0이라 가중치 업데이트가 미미하고,
Loss는 softmax 확률 전체의 미세한 변화로 줄어들 수 있지만
argmax로 결정되는 예측 클래스는 바뀌지 않아 Accuracy가 정체된다.
개선: BatchNorm을 추가하면 각 레이어 입력을 정규화해 Sigmoid가 포화 구간에서
벗어나도록 유도할 수 있고, 근본적으로는 ReLU 계열로 교체하는 것이 효과적이다.

---

### 공통 질문 3) Layer별 Activation 분포 변화

활성화 분포 히스토그램(셀9)에서:
- **ReLU 초반(Epoch 1):** 분포가 0에 과도하게 집중 (Dead ReLU 40.2%)
  기전: std=0.01 초기화로 pre-activation이 거의 0 → ReLU(x<0)=0 → Dead 유도
- **ReLU 후반(Epoch 300):** Dead 비율 20.9%로 감소 (초반 대비 절반 수준으로 회복)
  생존 뉴런만이 특정 방향을 학습하며 분포가 양수 쪽으로 이동하거나 고착됨
- **LeakyReLU:** 음수 구간에도 $\alpha x$ 값이 분포 → Dead 비율이 ReLU보다 낮음
  gradient가 0이 되지 않아 음수 영역 뉴런도 계속 학습에 기여
- **Sigmoid:** 학습 후반에 출력이 0 또는 1 근처로 몰림 → Saturation 비율 0.0% (make_moons처럼 단순한 데이터에서는 포화가 덜 발생)
  이 구간에서 $\sigma'(x) \approx 0$으로 gradient가 소멸

---

### 공통 질문 4) Gradient Vanishing/Exploding 영향

Gradient Norm 그래프(셀11)에서 Sigmoid의 Layer1 gradient norm(초반 0.00103)이
Layer3(초반 0.52506)보다 약 500배 작게 나타나는 것이 Vanishing Gradient의 직접 증거다.
기전: 역전파 gradient는 출력층에서 입력층 방향으로 전달되며,
각 레이어를 지날 때마다 $\frac{\partial L}{\partial h_l} = \frac{\partial L}{\partial h_{l+1}} \cdot \sigma'(x_l)$로 곱해진다.
$\sigma'(x) \leq 0.25$이므로 레이어 $L$개를 거치면 gradient 크기가 $0.25^L$로 감쇠.
결과적으로 앞쪽(입력에 가까운) 레이어는 gradient를 거의 받지 못해 학습이 멈춘다.

---

### 실험 B 질문 1) Dead ReLU가 학습에 미치는 영향

Dead ReLU 비율 그래프에서 ReLU의 Layer1 초반 비율은 40.2%였다.
Dead 뉴런이 학습을 정체시키는 기전:
- Dead 뉴런은 pre-activation < 0이므로 출력 = 0, gradient = 0
- 역전파 시 해당 뉴런의 gradient $\frac{\partial L}{\partial w} = 0$이 되어 가중치가 업데이트되지 않음
- 한 번 Dead 상태에 빠지면 어떤 입력이 들어와도 gradient가 0이라 영구적으로 Dead 유지
- 결과적으로 실질적으로 동작하는 뉴런 수가 줄어 모델의 표현력(capacity)이 감소하고
  학습 곡선이 정체되는 현상으로 나타난다

---

### 실험 B 질문 2) LeakyReLU의 Dead ReLU 완화 효과

LeakyReLU 정의:
$$f(x) = \begin{cases} x & x > 0 \\ \alpha x & x \leq 0 \end{cases}, \quad \alpha = 0.01$$

gradient:
$$f'(x) = \begin{cases} 1 & x > 0 \\ \alpha & x \leq 0 \end{cases}$$

음수 입력에서도 기울기 $\alpha = 0.01 > 0$이 유지되므로
역전파 gradient가 0이 되지 않고 $\alpha \cdot \frac{\partial L}{\partial \text{out}}$만큼 흘러 Dead 상태 탈출 가능.
실험에서 LeakyReLU의 최종 Dead 비율은 0.0%로, ReLU(20.9%)보다 현저히 낮게 나타났다.
히트맵에서도 LeakyReLU의 어두운 영역(비활성 뉴런)이 ReLU보다 현저히 적음을 확인.

---

### 실험 B 질문 3) Sigmoid Vanishing Gradient 발생 구간

Sigmoid gradient:
$$\sigma'(x) = \sigma(x)(1 - \sigma(x)), \quad \max = 0.25 \text{ (x=0에서)}$$

Vanishing 발생 기전:
- x가 양의 큰 값 또는 음의 큰 값으로 가면 $\sigma(x) \to 1$ 또는 $\sigma(x) \to 0$
- 이때 $\sigma'(x) = \sigma(1-\sigma) \to 0$으로 gradient가 소멸
- 레이어 2개만 거쳐도 $0.25^2 = 0.0625$, 레이어 $L$개면 $0.25^L \to 0$으로 지수 감쇠

Saturation 비율이 0.0%로 측정됐으나 Layer1 gradient norm이 초반 0.00103으로 거의 0에 가까웠고,
Accuracy 그래프에서 이 구간에 학습이 정체되는 패턴이 일치.
분포 히스토그램에서 후반으로 갈수록 출력이 0 또는 1 근처에 몰리는 것이 시각적 증거.

---

### 실험 B 질문 4) 각 Layer의 학습 기여도 히트맵 분석

히트맵(셀10)에서 공통 vmax 기준으로:
- **ReLU:** 어두운 영역(평균 활성화 ≈ 0)이 20.9%를 차지 → 해당 뉴런은 학습 기여도 없음
  특히 Layer1에서 비활성 뉴런이 집중되어 있어, 입력 정보의 상당 부분이 초반부터 차단됨
- **LeakyReLU:** 전반적으로 밝고 고른 분포 → 256개 뉴런이 골고루 학습에 기여
  공통 vmax 기준으로 ReLU보다 평균 활성화 값이 높아 더 많은 뉴런이 정보를 전달함

---

### 결론 및 개선 방안

| 활성화 함수 | Dead/Sat 비율 | 최종 정확도 | 90% 도달 Epoch |
|---|---|---|---|
| ReLU | Dead 20.9% | 98.00% | 25에폭 |
| LeakyReLU | Dead 0.0% | 98.50% | 21에폭 |
| Sigmoid | Sat 0.0% | 85.00% | 미달 |

**개선 방안:**
- Dead ReLU가 심각하면 → LeakyReLU / ELU 사용, 또는 He initialization(`std=√(2/fan_in)`)으로 교체
- Sigmoid Vanishing이 심하면 → BatchNorm 추가로 입력 분포 정규화, 또는 ReLU 계열로 교체
- 깊은 네트워크에서 Sigmoid는 사실상 사용 불가 → 현대 딥러닝에서 ReLU 계열이 기본값
